In [ ]:
"""
Combine_and_Analyze.py -- run this in Colab, same session where your Drive
is already mounted (or add the drive.mount() block below if it's a fresh
session).

What this does:
  1. Loads all 6 participants' trial_summary.csv files into one combined
     table (one row per participant x robot_name -- 30 rows total).
  2. Applies the Robot4 cross-participant exclusion (Participant 1) that's
     already documented in your Limitations chapter -- excluded rows are
     kept in the combined file (flagged, not deleted) but dropped from the
     cross-participant comparisons below, per your existing design.
  3. Produces descriptive statistics per robot condition (mean, std, N)
     for RMSSD delta and AU4/5/7/9 -- the first thing your Analytical
     Approach section calls for.
  4. Runs within-subject (repeated-measures) comparisons across the five
     conditions for RMSSD delta and AU4/5/7/9, using a non-parametric test
     (Friedman) appropriate for a small sample, rather than assuming
     normality.
  5. Runs correlations between each physiological/facial measure and each
     Godspeed subscale, across all included trials.
  6. Repeats steps 3-5 for AU1, AU2, and AU20 (fear-associated per the
     standard FACS investigator's guide, per Py-Feat's AU Reference page)
     as a SEPARATE, EXPLICITLY LABELED EXPLORATORY analysis -- kept fully
     separate from the primary AU4/5/7/9 results above so the a priori
     status of the primary analysis (already stated in your Methodology
     chapter) isn't blurred by adding AUs after already having looked at
     results. Report these, if at all, as exploratory/hypothesis-generating,
     not as part of the confirmatory core.
  7. Saves the combined dataset and results summaries back to Drive.

This is a FIRST LOOK, not your final Results chapter -- it tells you
whether there's anything there worth writing about, and gives you real
numbers to start interpreting. Read the printed output together with your
actual videos/conditions in mind before drawing conclusions.
"""

import pandas as pd
import numpy as np
from scipy import stats

# ==========================================
# Google Drive setup (skip if already mounted this session)
# ==========================================
from google.colab import drive
import os

drive.mount('/content/drive')
BASE_DIR = '/content/drive/MyDrive/ThesisExperiment_cleann'
os.chdir(BASE_DIR)
print(f"Working directory set to: {os.getcwd()}")

# ==========================================
# CONFIGURATION
# ==========================================
# Participant 2 excluded: no PPI data was recorded for their session
# (likely a mid-trial Bluetooth disconnection from the Polar sensor --
# see Limitations, Sample Size). Since this thesis's analysis depends on
# fused physiological+facial data, no attempt is made to salvage
# facial-only data for this participant.
PARTICIPANT_IDS = [1, 3, 4, 5, 6]

# Primary, a priori AUs -- selected on theoretical grounds before any
# detector comparison or result was looked at (see Methodology chapter).
KEY_AUS = ["AU04_mean", "AU05_mean", "AU07_mean", "AU09_mean"]

# Exploratory additions -- both added after the primary analysis had
# already been run, kept in named groups (never merged into KEY_AUS) so
# it's always clear, in the output and in anything copied into the
# thesis, which AUs were pre-registered vs. added afterward, and which
# emotion construct each addition relates to.
#
#   Fear-associated: AU1 (inner brow raiser), AU2 (outer brow raiser),
#   AU20 (lip stretcher) -- AU4 and AU5, also part of this same standard
#   FACS mapping, are already covered by the primary AUs above.
#
#   Disgust-associated: AU10 (upper lip raiser), AU17 (chin raiser) --
#   AU9, the third member of this mapping, is already one of the primary
#   AUs (and is already described in this thesis's own Methodology
#   chapter as "the expression of disgust"), so this addition rounds out
#   the same construct the primary analysis is already anchored to,
#   rather than introducing an unrelated one. AU10 was previously
#   considered for this group but removed: Py-Feat's own AU Reference
#   table lists no emotional association for AU10 at all (its inclusion
#   would have relied on the older, separate FACS investigator's-guide
#   combination convention, a different source than the rest of this
#   thesis's AU-selection reasoning). Restricting every AU group in this
#   analysis to associations Py-Feat's own documentation lists directly
#   keeps the whole selection process traceable to one single, checkable
#   source (py-feat.org/pages/au_reference).
#
#   The "stress-associated" group used in earlier drafts of this script
#   (AU6, AU12, AU14, AU15, AU23, AU25, AU26, drawn from Giannakakis et
#   al., 2024) has been removed for the same reason: that source is not
#   accessible to re-verify, unlike Py-Feat's own table. See Limitations
#   for the resulting AU45/blink coverage-gap note, which still applies
#   independently of this group's removal.
EXPLORATORY_GROUPS = {
    "Fear-associated (AU1, AU2, AU20)": ["AU01_mean", "AU02_mean", "AU20_mean"],
    "Disgust-associated (AU17)": ["AU17_mean"],
}

AFFECT_COLS = ["arousal_mean", "valence_mean"]
GODSPEED_COLS = ["anthropomorphism", "animacy", "likeability",
                  "perceived_intelligence", "perceived_safety"]

# ==========================================
# STEP 1: Load and combine all 6 trial summaries
# ==========================================
print("\nLoading trial summaries for all participants...")
all_summaries = []
for pid in PARTICIPANT_IDS:
    fname = f"participant_{pid}_trial_summary.csv"
    if not os.path.exists(fname):
        print(f"  [!] WARNING: {fname} not found, skipping participant {pid}.")
        continue
    df = pd.read_csv(fname)
    all_summaries.append(df)
    print(f"  [+] Loaded {fname} ({len(df)} rows)")

combined = pd.concat(all_summaries, ignore_index=True)
print(f"\nCombined dataset: {len(combined)} rows "
      f"({combined['participant_id'].nunique()} participants x "
      f"{combined['robot_name'].nunique()} conditions)")

# Save the full combined file (including excluded rows) for your records
combined_output = "all_participants_trial_summary.csv"
combined.to_csv(combined_output, index=False)
print(f"Saved combined dataset to: {combined_output}")

# ==========================================
# STEP 2: Apply cross-participant exclusions
# ==========================================
if "excluded_cross_participant" in combined.columns:
    n_excluded = combined["excluded_cross_participant"].sum()
    print(f"\n{n_excluded} row(s) flagged as excluded_cross_participant "
          f"(e.g. Participant 1's Robot4 trial, per Limitations) -- "
          f"kept in the saved file above, dropped from analysis below.")
    analysis_df = combined[combined["excluded_cross_participant"] == False].copy()
else:
    print("\n  [!] WARNING: no 'excluded_cross_participant' column found -- "
          "proceeding with all rows.")
    analysis_df = combined.copy()

# ==========================================
# STEP 3: Descriptive statistics per condition -- PRIMARY (a priori AUs)
# ==========================================
print("\n" + "=" * 70)
print("PRIMARY ANALYSIS (a priori) -- DESCRIPTIVE STATISTICS BY CONDITION")
print("=" * 70)

desc_cols = ["rmssd_delta"] + KEY_AUS + AFFECT_COLS
desc_stats = analysis_df.groupby("robot_name")[desc_cols].agg(["mean", "std", "count"])
print(desc_stats.round(3).to_string())

# ==========================================
# STEP 4: Within-subject comparison across conditions -- PRIMARY (Friedman test)
# ==========================================
# Friedman is the non-parametric equivalent of repeated-measures ANOVA --
# appropriate here given the small sample size flagged in your Limitations
# (Analytical Approach), which rules out assuming normality.
print("\n" + "=" * 70)
print("PRIMARY ANALYSIS -- WITHIN-SUBJECT COMPARISON ACROSS CONDITIONS (Friedman test)")
print("=" * 70)
print("(Requires each participant to have all 5 conditions present;")
print(" participants missing a condition due to exclusion are dropped")
print(" from that specific test.)\n")

robot_order = ["Robot1", "Robot2", "Robot3", "Robot4", "Human"]

for measure in ["rmssd_delta"] + KEY_AUS + AFFECT_COLS:
    pivot = analysis_df.pivot_table(index="participant_id", columns="robot_name",
                                      values=measure)
    pivot = pivot.reindex(columns=robot_order).dropna()
    if len(pivot) < 2:
        print(f"{measure}: skipped (fewer than 2 participants with complete data)")
        continue
    stat, p = stats.friedmanchisquare(*[pivot[c] for c in robot_order])
    sig = "*" if p < 0.05 else " "
    print(f"{measure:20s} chi2={stat:6.2f}  p={p:.4f} {sig}  (N={len(pivot)} participants)")

print("\n(* p < .05 -- with N this small, treat any significant result as")
print(" hypothesis-generating, per your Limitations chapter, not confirmatory.)")

# ==========================================
# STEP 5: Correlations with Godspeed subscales -- PRIMARY
# ==========================================
print("\n" + "=" * 70)
print("PRIMARY ANALYSIS -- CORRELATIONS vs. Godspeed Subscales")
print("=" * 70)

corr_vars = ["rmssd_delta"] + KEY_AUS + AFFECT_COLS
corr_results = []
for var in corr_vars:
    for god in GODSPEED_COLS:
        paired = analysis_df[[var, god]].dropna()
        if len(paired) < 3:
            continue
        r, p = stats.pearsonr(paired[var], paired[god])
        corr_results.append({"measure": var, "godspeed_subscale": god,
                              "r": round(r, 3), "p": round(p, 4), "N": len(paired)})

corr_df = pd.DataFrame(corr_results).sort_values("p")
print(corr_df.to_string(index=False))

# ==========================================
# STEP 6: EXPLORATORY analysis -- fear- and disgust-associated AUs
# ==========================================
# Everything in this block is EXPLORATORY, added after the primary
# analysis above had already been run. Report it as hypothesis-generating
# if at all -- do not merge these results into the primary AU4/5/7/9
# findings or imply they were part of the original a priori plan, even
# where (as with the disgust-associated AUs) the theoretical link to the
# thesis's existing framing is close.
print("\n" + "#" * 70)
print("# EXPLORATORY ANALYSIS -- added after primary results were already")
print("# examined. Not part of the original a priori plan. Report as")
print("# hypothesis-generating, not confirmatory.")
print("#" * 70)

exp_desc_frames = {}
exp_corr_frames = {}

for group_label, group_aus in EXPLORATORY_GROUPS.items():
    print("\n" + "=" * 70)
    print(f"EXPLORATORY -- {group_label} -- DESCRIPTIVE STATISTICS BY CONDITION")
    print("=" * 70)
    group_desc = analysis_df.groupby("robot_name")[group_aus].agg(["mean", "std", "count"])
    print(group_desc.round(3).to_string())
    exp_desc_frames[group_label] = group_desc

    print("\n" + "=" * 70)
    print(f"EXPLORATORY -- {group_label} -- WITHIN-SUBJECT COMPARISON (Friedman test)")
    print("=" * 70)
    for measure in group_aus:
        pivot = analysis_df.pivot_table(index="participant_id", columns="robot_name",
                                          values=measure)
        pivot = pivot.reindex(columns=robot_order).dropna()
        if len(pivot) < 2:
            print(f"{measure}: skipped (fewer than 2 participants with complete data)")
            continue
        stat, p = stats.friedmanchisquare(*[pivot[c] for c in robot_order])
        sig = "*" if p < 0.05 else " "
        print(f"{measure:20s} chi2={stat:6.2f}  p={p:.4f} {sig}  (N={len(pivot)} participants)")

    print("\n" + "=" * 70)
    print(f"EXPLORATORY -- {group_label} -- CORRELATIONS vs. Godspeed Subscales")
    print("=" * 70)
    group_corr_results = []
    for var in group_aus:
        for god in GODSPEED_COLS:
            paired = analysis_df[[var, god]].dropna()
            if len(paired) < 3:
                continue
            r, p = stats.pearsonr(paired[var], paired[god])
            group_corr_results.append({"measure": var, "godspeed_subscale": god,
                                        "r": round(r, 3), "p": round(p, 4), "N": len(paired)})
    group_corr_df = pd.DataFrame(group_corr_results).sort_values("p")
    print(group_corr_df.to_string(index=False))
    exp_corr_frames[group_label] = group_corr_df

# ==========================================
# STEP 7: Save results
# ==========================================
corr_output = "correlation_results_primary.csv"
corr_df.to_csv(corr_output, index=False)
desc_output = "descriptive_stats_by_condition_primary.csv"
desc_stats.to_csv(desc_output)

saved_exploratory = [combined_output, desc_output, corr_output]
for group_label, group_desc in exp_desc_frames.items():
    slug = group_label.split(" (")[0].lower().replace(" ", "_").replace("-", "_")
    fname = f"descriptive_stats_exploratory_{slug}.csv"
    group_desc.to_csv(fname)
    saved_exploratory.append(fname)
for group_label, group_corr_df in exp_corr_frames.items():
    slug = group_label.split(" (")[0].lower().replace(" ", "_").replace("-", "_")
    fname = f"correlation_results_exploratory_{slug}.csv"
    group_corr_df.to_csv(fname, index=False)
    saved_exploratory.append(fname)

print(f"\nSaved (primary): {combined_output}, {desc_output}, {corr_output}")
print(f"Saved (exploratory): {', '.join(saved_exploratory[3:])}")
print("\nAll outputs saved to your Drive folder alongside your other files.")